In [ ]:
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
 
import polars as pl
from Bio import SeqIO
 
sys.path.insert(0, "..")
from project import check_run_settings, get_all, get_info, log, tool, update_run_json

In [ ]:
ALL = get_all()
OVERWRITE = False  # True re-aligns every gene pair and allows changing the settings of an existing run
WORKERS = 32  # one gene pair per worker, Clustal Omega with 1 thread each
 
CLUSTALO = tool("clustalo")
if shutil.which(CLUSTALO) is None:
    raise FileNotFoundError(f"'{CLUSTALO}' not found in PATH")
clustalo_version = subprocess.run([CLUSTALO, "--version"], capture_output=True, text=True).stdout.strip()
print(f"Clustal Omega: {clustalo_version}")

In [ ]:
ALL = {
    "plants_plt": {
        "solanaceae": get_info("plants_plt", "solanaceae"),
    }
}

In [ ]:
def build_cmd(in_fasta, out_fasta, log_file) -> list[str]:
    return [
        CLUSTALO,
        "-i", str(in_fasta),
        "-o", str(out_fasta),
        "--outfmt=fa",
        "-t", "DNA",
        "--full-iter",
        "--iter=3",
        "--output-order=input-order",
        "--threads=1",
        f"--log={log_file}",
        "--verbose",
        "--force",
    ]
 
 
def run_clustalo(gp: str, igr_dir: Path, run_dir: Path, logs_dir: Path) -> tuple[str, str]:
    in_fasta = igr_dir / f"gp_{gp}.fasta"
    out_fasta = run_dir / f"gp_{gp}_aligned.fasta"
    log_file = logs_dir / f"gp_{gp}_clustalo.log"
 
    if out_fasta.exists() and not OVERWRITE:
        return gp, "skipped"
 
    tmp = out_fasta.with_suffix(".tmp")  # never leave a half-written alignment behind
    res = subprocess.run(build_cmd(in_fasta, tmp, log_file), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.returncode != 0 or not tmp.exists():
        (logs_dir / f"gp_{gp}_clustalo_error.log").write_text(res.stdout)
        tmp.unlink(missing_ok=True)
        return gp, f"error (exit {res.returncode})"
    tmp.replace(out_fasta)
    return gp, "ok"

In [ ]:
for g, subs in ALL.items():
    for subg, info in subs.items():
        p = info["paths"]
        clustalo_run = info["runs"]["clustalo"]
        igr_dir = p["igr_by_pair"]
        run_dir = p["alignments"] / clustalo_run
        logs_dir = run_dir / "logs"
        run_json = run_dir / "run.json"
        log(f"{g}/{subg} | clustalo run: {clustalo_run}")
 
        if not p["gene_pairs_filtered"].exists():
            log(f"  no {p['gene_pairs_filtered'].name}, run 02_gene_pairs first; skipping")
            continue
 
        template = build_cmd("<igr_by_pair>/gp_{gp}.fasta", "<run_dir>/gp_{gp}_aligned.fasta", "<logs_dir>/gp_{gp}_clustalo.log")
        settings = {"command_template": " ".join(template)}
        check_run_settings(run_json, settings, OVERWRITE)
 
        # --- every kept gene pair needs an input FASTA with at least 2 sequences
        gp_df = pl.read_csv(p["gene_pairs_filtered"], separator="\t")
        problems = []
        for gp in gp_df["gene_pair"]:
            fasta = igr_dir / f"gp_{gp}.fasta"
            if not fasta.exists():
                problems.append(f"{gp}: missing {fasta.name}")
            elif sum(1 for _ in SeqIO.parse(fasta, "fasta")) < 2:
                problems.append(f"{gp}: fewer than 2 sequences")
        if problems:
            log(f"  {len(problems)} of {gp_df.height} gene pairs have problems:")
            for prob in problems:
                print(f"    {prob}")
            raise RuntimeError(f"{g}/{subg}: input FASTAs are missing or too small; nothing was aligned.")
 
        # largest jobs first, so the long ones don't end up running alone at the end
        pairs = gp_df.sort(pl.col("igr_count") * pl.col("mean_len"), descending=True)["gene_pair"].to_list()
        logs_dir.mkdir(parents=True, exist_ok=True)
 
        update_run_json(
            run_json,
            run_name=clustalo_run,
            group=g,
            subgroup=subg,
            status="running",
            date_started=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            date_finished=None,
            **settings,
            clustalo=CLUSTALO,
            clustalo_path=shutil.which(CLUSTALO),
            clustalo_version=clustalo_version,
            input_fastas=str(igr_dir.relative_to(p["dir"])),
            gene_pairs_file=str(p["gene_pairs_filtered"].relative_to(p["dir"])),
            n_gene_pairs=len(pairs),
        )
 
        # --- align
        t0 = time.time()
        counts: dict[str, int] = {}
        errors = []
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            tasks = [executor.submit(run_clustalo, gp, igr_dir, run_dir, logs_dir) for gp in pairs]
            for i, future in enumerate(as_completed(tasks), 1):
                gp, status = future.result()
                key = "error" if status.startswith("error") else status
                counts[key] = counts.get(key, 0) + 1
                if key == "error":
                    errors.append(f"{gp}: {status}")
                    log(f"  [ERROR] {gp}: {status}")
                if i % 25 == 0 or i == len(tasks):
                    log(f"    {i}/{len(tasks)} done | {counts}")
 
        runtime = time.time() - t0
        update_run_json(
            run_json,
            status="complete" if not errors else "complete_with_errors",
            date_finished=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            runtime_sec=round(runtime, 1),
            counts=counts,
            errors=errors,
        )
 
        stale = {f.name.removeprefix("gp_").removesuffix("_aligned.fasta") for f in run_dir.glob("gp_*_aligned.fasta")} - set(pairs)
        log(f"  {counts} | runtime {int(runtime // 60)}m{int(runtime % 60):02d}s -> {run_dir.relative_to(p['dir'])}/")
        if stale:
            log(f"  note: {len(stale)} alignments in this run are for gene pairs no longer kept: {sorted(stale)}")
 
log("Done")

(old version, 03b)

plants_plt
    runtime: 55m27s
    pairs: 195

plants_mit
    runtime: 7m8s
    pairs: 32